In [ ]:
import random
import time
import math
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "cross-encoder/stsb-distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
subset_size = 500
max_length = 256
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 32 if device == "mps" else 16
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "subset_size": subset_size,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)
df = df.head(subset_size).reset_index(drop=True)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
pipe_device = device
clf = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipe_device,
    truncation=True,
    max_length=max_length,
)

print({
    "pipeline_task": "text-classification",
    "model_name": model_name,
    "device": pipe_device,
})

In [ ]:
pairs = [{"text": s1, "text_pair": s2} for s1, s2 in zip(df["sentence1"], df["sentence2"])]

outputs = clf(
    pairs,
    batch_size=batch_size,
    truncation=True,
    max_length=max_length,
)

predicted_scores = np.array([float(x["score"]) for x in outputs], dtype=np.float32)
labels = df["label"].to_numpy(dtype=np.float32)
residuals = predicted_scores - labels
absolute_errors = np.abs(residuals)
squared_errors = residuals ** 2

print(predicted_scores[:10])

In [ ]:
pearson_corr = pearsonr(predicted_scores, labels).statistic
spearman_corr = spearmanr(predicted_scores, labels).statistic
mae = float(np.mean(absolute_errors))
rmse = float(np.sqrt(np.mean(squared_errors)))

results_df = df.copy()
results_df["predicted_score"] = predicted_scores
results_df["residual"] = residuals
results_df["absolute_error"] = absolute_errors

score_mean = float(np.mean(predicted_scores))
score_std = float(np.std(predicted_scores))
label_mean = float(np.mean(labels))
label_std = float(np.std(labels))

print(results_df[["sentence1", "sentence2", "label", "predicted_score", "residual", "absolute_error"]].head(10))

In [ ]:
top_k = 5

highest_residual_examples = results_df.nlargest(top_k, "residual")[[
    "sentence1", "sentence2", "label", "predicted_score", "residual", "absolute_error"
]].reset_index(drop=True)

lowest_residual_examples = results_df.nsmallest(top_k, "residual")[[
    "sentence1", "sentence2", "label", "predicted_score", "residual", "absolute_error"
]].reset_index(drop=True)

largest_absolute_residual_examples = results_df.nlargest(top_k, "absolute_error")[[
    "sentence1", "sentence2", "label", "predicted_score", "residual", "absolute_error"
]].reset_index(drop=True)

print("Highest residual examples (over-predictions):")
print(highest_residual_examples.to_string(index=False))
print()
print("Lowest residual examples (under-predictions):")
print(lowest_residual_examples.to_string(index=False))
print()
print("Largest absolute residual examples:")
print(largest_absolute_residual_examples.to_string(index=False))

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"subset_size: {len(df)}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"mae: {mae:.6f}")
print(f"rmse: {rmse:.6f}")
print(f"predicted_score_mean: {score_mean:.6f}")
print(f"predicted_score_std: {score_std:.6f}")
print(f"label_mean: {label_mean:.6f}")
print(f"label_std: {label_std:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")